In [63]:
'''

We perform two calibration processes:
    
    One long calibration, where we calibrate with
        Co-60, Na (preamp 100)
        Ba, Cs, Co-57 (preamp 0; Co-57 done overnight)
    this long calibration is done only once (Feb 24; Co-57 run was done 5:45 PM Feb 24-3PM Feb 25).

    *Every* time we take data, we will do a single Cs (preamp level 0) and a Co-60 (preamp level 100) run to 
    see the variation in our ADC output compared to the previous baseline.

    In our final draft these calibrations have to be made to account for the background. The background analysis is also done here.
    
'''

'\n\nWe perform two calibration processes:\n\n    One long calibration, where we calibrate with\n        Co-60, Na (preamp 100)\n        Ba, Cs, Co-57 (preamp 0; Co-57 done overnight)\n    this long calibration is done only once (Feb 24; Co-57 run was done 5:45 PM Feb 24-3PM Feb 25).\n\n    *Every* time we take data, we will do a single Cs (preamp level 0) and a Co-60 (preamp level 100) run to \n    see the variation in our ADC output compared to the previous baseline.\n\n    In our final draft these calibrations have to be made to account for the background. The background analysis is also done here.\n\n'

In [64]:
'''

Naming schema: notation.

I denote file folders with [] and nuclear data files with {}

'''

'\n\nNaming schema: notation.\n\nI denote file folders with [] and nuclear data files with {}\n\n'

In [65]:
'''

ROUGH FILE STRUCTURE

[] calibration
    [] feb_24
        [] preamp_100
            {} Co60 _22.5C_preamp100_feb24
            {} Na_22.5C_preamp100_feb24
        [] preamp_0
            {} Ba_22.4C_preamp0_feb24
            {} Cs_22.1C_preamp0_feb24
    [] daily
        {} Cs_22.4C_preamp0_feb26
        {} Co60_22.4C_preamp100_feb26

'''

'\n\nROUGH FILE STRUCTURE\n\n[] calibration\n    [] feb_24\n        [] preamp_100\n            {} Co60 _22.5C_preamp100_feb24\n            {} Na_22.5C_preamp100_feb24\n        [] preamp_0\n            {} Ba_22.4C_preamp0_feb24\n            {} Cs_22.1C_preamp0_feb24\n    [] daily\n        {} Cs_22.4C_preamp0_feb26\n        {} Co60_22.4C_preamp100_feb26\n\n'

In [66]:
'''

Goal:
    Subtract background data from our data fit
    
    Generate a calibration curve from the big run (Feb 24)
        There should be two such curves, one for preamp level 0 and preamp level 100
    
    Generate a data algorithm which adjusts our day-by-day analysis for changes in ADC output
        There should be two such curves, one for preamp level 0 and preamp level 100
    Any change from the big run should be roughly on the order of 13 ADC channels (quite small, but worth accounting for anyways)
    We want to do this by forcefully changing the slope (?) or something from the original calibration curve

    This code should produce a calibration curve graph for our feb 24 run, as well as our night runs for co-57 
    (using whatever schema we choose to use to account for the calibration curve of these data points)

    Every trial we take, with the exception of the background run, is 15 minutes.
    
'''

'\n\nGoal:\n    Subtract background data from our data fit\n\n    Generate a calibration curve from the big run (Feb 24)\n        There should be two such curves, one for preamp level 0 and preamp level 100\n\n    Generate a data algorithm which adjusts our day-by-day analysis for changes in ADC output\n        There should be two such curves, one for preamp level 0 and preamp level 100\n    Any change from the big run should be roughly on the order of 13 ADC channels (quite small, but worth accounting for anyways)\n    We want to do this by forcefully changing the slope (?) or something from the original calibration curve\n\n    This code should produce a calibration curve graph for our feb 24 run, as well as our night runs for co-57 \n    (using whatever schema we choose to use to account for the calibration curve of these data points)\n\n    Every trial we take, with the exception of the background run, is 15 minutes.\n\n'

In [67]:
# -----------------------------
# IMPORT STATEMENTS
# -----------------------------

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import find_peaks, savgol_filter
from scipy.optimize import curve_fit

In [68]:
# -----------------------------
# CONFIG: USER SETUP (!!!)
# -----------------------------

# Root directory where converted CSV calibration data live
ROOT_CAL = Path("processed_data/calibration")  # <-- change if your path differs

# Background file to subtract during calibration (or leave None)
BG_DEFAULT = Path("processed_data/calibration/background_550PM_feb26_213PM_mar3.csv")

CFG = {
    # Peak finding
    "smooth_window": 21,          # savgol window (odd)
    "smooth_poly": 3,             # savgol polyorder
    "min_distance": 20,           # minimum distance between peaks (channels)
    "prominence_frac": 0.02,      # prominence threshold as fraction of max(y) in spectrum
    "max_peaks_global": 40,       # how many global peaks to attempt fitting (baseline discovery)

    # Peak fitting windows
    "fit_half_window": 50,        # +/- window around candidate peak for Gaussian fit
    "fit_min_points": 25,         # minimum points required in window
    "sigma_bounds": (1.0, 200.0), # allowed sigma range in channels (tune!)

    # Fit QA
    "max_chi2_ndf": 10.0,
    "min_amp_snr": 3.0,           # A / dA threshold (rough)
    "require_positive_amp": True,

    # Calibration model (usually linear is enough for MCA)
    "cal_model": "linear",        # "linear" or "quadratic"

    # Drift tracking windows (for tracking baseline peaks)
    "track_search_half_window": 80,   # how far to search around baseline mu
    "track_fit_half_window": 50,      # fit window once candidate selected

    # Daily drift model
    # preamp0: shift-only (Cs 661)
    # preamp100: affine if both Co60 lines found, else shift-only fallback
}

In [69]:
# -----------------------------
# CONFIG: KNOWN SPECTROSCOPY DATA
# -----------------------------

'''
Using Idaho National Labratory's NaI Gamma spec data

All listed values are in keV
'''

# -----------------------------
# PREAMP 0 
# -----------------------------

# --- used for daily calibration curve ---
# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/cs137.pdf
Cs_spectra     = np.array([31.8, 37.3, 661.657])
Cs_spectra_err = np.array([0,    0,    0.003])
# ---                                  ---

# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/ba133.pdf
Ba_spectra     = np.array([53.15, 79.60, 80.998, 160.605, 223.246, 276.397, 302.851, 356.005, 383.851])
Ba_spectra_err = np.array([0.05,  0.05,  0.008,  0.015,   0.030,   0.012,   0.015,   0.017,   0.020])



# -----------------------------
# PREAMP 100 
# -----------------------------

# --- used for daily calibration curve ---
# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/co60.pdf
Co60_spectra     = np.array([1173.228, 1332.494])
Co60_spectra_err = np.array([0.003,    0.002])
# ---                                  ---

# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/na22.pdf
Na_spectra     = np.array([511.006, 1274.537])
Na_spectra_err = np.array([0,       0.008])


# -----------------------------
# OVERNIGHT RUNS: Co-57 (set to preamp 0) 
# -----------------------------
'''
for overnight runs day-by-day calibration curves will no longer be accurate so
we will need to use some sort of average of calibration curve data for every day
which accounts for the ncreased error
'''

# see https://gammaray.inl.gov/SiteAssets/catalogs/nai/pdf/co57.pdf
Co57_spectra     = np.array([14.4136, 122.060, 136.471, 570.1, 692.4])
Co57_spectra_err = np.array([0.0003,  0.010,   0.010,   0.2,   0.1])

In [70]:
# -----------------------------
# DATA STRUCTURES
# -----------------------------

@dataclass
class Spectrum:
    path: Path
    channel: np.ndarray
    counts: np.ndarray
    meta: dict

    @property
    def dt_seconds(self) -> float:
        return float(self.meta.get("elapsed_real_time_s") or self.meta.get("dt_seconds") or np.nan)

@dataclass
class NetSpectrum:
    # background-subtracted (in counts) + propagated errors
    path_sig: Path
    channel: np.ndarray
    net_counts: np.ndarray
    net_sigma: np.ndarray
    meta_sig: dict
    meta_bg: dict | None

    @property
    def dt_sig(self) -> float:
        return float(self.meta_sig.get("elapsed_real_time_s") or self.meta_sig.get("dt_seconds") or np.nan)

@dataclass
class PeakFit:
    mu: float
    mu_err: float
    sigma: float
    sigma_err: float
    amp: float
    amp_err: float
    c0: float
    c1: float
    chi2: float
    ndf: int
    success: bool
    reason: str = ""

    @property
    def chi2_ndf(self) -> float:
        return self.chi2 / self.ndf if self.ndf > 0 else np.inf

@dataclass
class CalibrationFit:
    model: str
    params: np.ndarray
    cov: np.ndarray
    chi2: float
    ndf: int
    used_points: int

    @property
    def chi2_ndf(self) -> float:
        return self.chi2 / self.ndf if self.ndf > 0 else np.inf

In [71]:
# -----------------------------
# CSV HEADER PROCESSING
# -----------------------------

_TIME_PAT = re.compile(r"^(Start Time|End Time)\s*:\s*(.*)$", re.IGNORECASE)
_ELAPSED_PAT = re.compile(r"^(Elapsed Real Time|Elapsed Live Time)\s*:\s*(.*)$", re.IGNORECASE)

def _parse_hms_to_seconds(s: str) -> float | None:
    s = s.strip()
    # Common forms: "00:15:00" or "0:15:00"
    m = re.match(r"^(\d+)\s*:\s*(\d+)\s*:\s*(\d+)\s*$", s)
    if not m:
        return None
    h, mi, se = map(int, m.groups())
    return float(h*3600 + mi*60 + se)

def _parse_datetime_fuzzy(s: str) -> datetime | None:
    s = s.strip()
    # Your files look like: "Thursday, February 26, 2026, 17:44:33"
    # We'll try a couple of formats.
    for fmt in [
        "%A, %B %d, %Y, %H:%M:%S",
        "%B %d, %Y, %H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
    ]:
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    return None

def read_spectrum_csv(path: Path) -> Spectrum:
    """
    Reads your processed CSV:
      - parses header lines (Start/End time, elapsed times)
      - reads the data table that begins after 'Channel Data:' and the column header line

    Returns Spectrum(channel, counts, meta)
    """
    text = path.read_text(errors="ignore").splitlines()

    meta: dict = {"path": str(path)}
    data_start_idx = None
    col_header_idx = None

    for i, line in enumerate(text):
        line_stripped = line.strip()

        # Start/End time
        tm = _TIME_PAT.match(line_stripped)
        if tm:
            key = tm.group(1).lower().replace(" ", "_")  # start_time / end_time
            dt = _parse_datetime_fuzzy(tm.group(2))
            meta[key] = dt
            continue

        # Elapsed time
        em = _ELAPSED_PAT.match(line_stripped)
        if em:
            label = em.group(1).lower().replace(" ", "_")  # elapsed_real_time / elapsed_live_time
            sec = _parse_hms_to_seconds(em.group(2))
            if sec is not None:
                meta[label + "_s"] = sec
            continue

        # Find where table starts
        if line_stripped.lower().startswith("channel data"):
            data_start_idx = i
            continue

        # Column header line typically "Channel,Energy,Counts" (Energy may be blank later)
        if data_start_idx is not None and col_header_idx is None:
            if "channel" in line_stripped.lower() and "counts" in line_stripped.lower():
                col_header_idx = i
                continue

        # First data row: starts with an integer channel followed by commas
        if col_header_idx is not None and i > col_header_idx:
            if re.match(r"^\s*\d+\s*,", line):
                # assume rest is data
                data_start_idx = col_header_idx
                break

    # Default dt_seconds if not explicitly provided
    if meta.get("elapsed_real_time_s") is None:
        st = meta.get("start_time")
        et = meta.get("end_time")
        if isinstance(st, datetime) and isinstance(et, datetime):
            meta["dt_seconds"] = (et - st).total_seconds()

    # Read table: pandas can handle skipping header block if we locate "Channel,Energy,Counts"
    # We’ll detect the line index of the column header and read from there.
    if col_header_idx is None:
        # fallback: try reading last 4096-ish rows as table
        df = pd.read_csv(path)
    else:
        # read from the header line
        df = pd.read_csv(path, skiprows=col_header_idx)

    # Standardize columns
    cols = [c.strip().lower() for c in df.columns]
    df.columns = cols

    # Expect at least channel and counts
    if "channel" not in df.columns:
        # try first col
        df = df.rename(columns={df.columns[0]: "channel"})
    if "counts" not in df.columns:
        # try last col
        df = df.rename(columns={df.columns[-1]: "counts"})

    df = df[["channel", "counts"]].copy()
    df["channel"] = pd.to_numeric(df["channel"], errors="coerce").astype("Int64")
    df["counts"] = pd.to_numeric(df["counts"], errors="coerce")

    df = df.dropna()
    channel = df["channel"].to_numpy(dtype=float)
    counts = df["counts"].to_numpy(dtype=float)

    return Spectrum(path=path, channel=channel, counts=counts, meta=meta)


In [72]:
# -----------------------------
# BACKGROUND RADIATION UTILITIES
# -----------------------------

def subtract_background_counts(sig: Spectrum, bg: Spectrum) -> NetSpectrum:
    """
    Net = sig - (t_sig/t_bg)*bg, per-channel.
    var = sig + (t_sig/t_bg)^2 * bg (Poisson)
    """
    if sig.channel.shape != bg.channel.shape or not np.allclose(sig.channel, bg.channel):
        raise ValueError("Signal and background channels don't match.")

    t_sig = sig.dt_seconds
    t_bg = bg.dt_seconds
    if not np.isfinite(t_sig) or not np.isfinite(t_bg) or t_sig <= 0 or t_bg <= 0:
        raise ValueError(f"Bad dt (t_sig={t_sig}, t_bg={t_bg}). Ensure elapsed time or start/end times parsed correctly.")

    scale = t_sig / t_bg
    net = sig.counts - scale * bg.counts
    var = sig.counts + (scale**2) * bg.counts
    sigma = np.sqrt(np.clip(var, 0, np.inf))

    return NetSpectrum(
        path_sig=sig.path,
        channel=sig.channel.copy(),
        net_counts=net,
        net_sigma=sigma,
        meta_sig=sig.meta,
        meta_bg=bg.meta,
    )

def spectrum_with_errors(spec: Spectrum, assume_poisson: bool = True) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = spec.channel
    y = spec.counts
    if assume_poisson:
        yerr = np.sqrt(np.clip(y, 0, np.inf))
    else:
        yerr = np.ones_like(y)
    return x, y, yerr

In [73]:
# -----------------------------
# PEAK FINDING & GAUSSIAN FIT
# -----------------------------

def _smooth(y: np.ndarray, window: int, poly: int) -> np.ndarray:
    window = int(window)
    if window < 5:
        return y.copy()
    if window % 2 == 0:
        window += 1
    if window >= len(y):
        window = max(5, (len(y)//2)*2 - 1)
    return savgol_filter(y, window_length=window, polyorder=min(poly, window-2))

def find_peaks_global(x: np.ndarray, y: np.ndarray, cfg: dict) -> np.ndarray:
    ys = _smooth(y, cfg["smooth_window"], cfg["smooth_poly"])
    prom = float(cfg["prominence_frac"]) * float(np.nanmax(ys) if np.nanmax(ys) > 0 else 1.0)
    peaks, props = find_peaks(ys, prominence=prom, distance=cfg["min_distance"])
    # Sort by prominence descending
    if len(peaks) == 0:
        return peaks
    order = np.argsort(props["prominences"])[::-1]
    peaks = peaks[order]
    return peaks[: cfg["max_peaks_global"]]

def gaussian_linear(x, A, mu, sig, c0, c1):
    return A * np.exp(-0.5 * ((x - mu)/sig)**2) + (c0 + c1*x)

def fit_peak_gaussian(x: np.ndarray, y: np.ndarray, yerr: np.ndarray, idx0: int, cfg: dict) -> PeakFit:
    half = int(cfg["fit_half_window"])
    L = max(0, idx0 - half)
    R = min(len(x)-1, idx0 + half)
    if R - L + 1 < cfg["fit_min_points"]:
        return PeakFit(0,0,0,0,0,0,0,0, np.inf, 0, False, "window too small")

    xw = x[L:R+1]
    yw = y[L:R+1]
    ew = yerr[L:R+1]
    # Avoid zero errors
    ew = np.where(ew > 0, ew, 1.0)

    # Initial guesses
    mu0 = x[idx0]
    # baseline from edges
    edge = max(3, len(yw)//10)
    baseline0 = float(np.median(np.r_[yw[:edge], yw[-edge:]]))
    amp0 = float(np.max(yw) - baseline0)
    if not np.isfinite(amp0) or amp0 == 0:
        amp0 = float(np.max(yw) if np.max(yw) != 0 else 1.0)
    sig0 = 10.0

    p0 = [amp0, mu0, sig0, baseline0, 0.0]

    # Bounds
    sig_lo, sig_hi = cfg["sigma_bounds"]
    # amplitude can be negative if net spectrum has negatives, but you can force positive if desired
    if cfg.get("require_positive_amp", True):
        A_bounds = (0.0, np.inf)
    else:
        A_bounds = (-np.inf, np.inf)

    bounds = (
        [A_bounds[0], xw.min(), sig_lo, -np.inf, -np.inf],
        [A_bounds[1], xw.max(), sig_hi,  np.inf,  np.inf],
    )

    try:
        popt, pcov = curve_fit(
            gaussian_linear, xw, yw, p0=p0, sigma=ew, absolute_sigma=True,
            bounds=bounds, maxfev=20000
        )
        A, mu, sig, c0, c1 = popt
        perr = np.sqrt(np.diag(pcov))
        Aerr, muerr, sigerr = perr[0], perr[1], perr[2]

        # chi-square
        resid = (yw - gaussian_linear(xw, *popt)) / ew
        chi2 = float(np.sum(resid**2))
        ndf = int(len(xw) - len(popt))

        # QA
        if not (xw.min() < mu < xw.max()):
            return PeakFit(mu, muerr, sig, sigerr, A, Aerr, c0, c1, chi2, ndf, False, "mu outside window")
        if cfg.get("require_positive_amp", True) and A <= 0:
            return PeakFit(mu, muerr, sig, sigerr, A, Aerr, c0, c1, chi2, ndf, False, "amp <= 0")
        if ndf <= 0:
            return PeakFit(mu, muerr, sig, sigerr, A, Aerr, c0, c1, chi2, ndf, False, "ndf<=0")
        if chi2/ndf > cfg["max_chi2_ndf"]:
            return PeakFit(mu, muerr, sig, sigerr, A, Aerr, c0, c1, chi2, ndf, False, "chi2/ndf too large")
        if np.isfinite(Aerr) and Aerr > 0:
            snr = abs(A) / Aerr
            if snr < cfg["min_amp_snr"]:
                return PeakFit(mu, muerr, sig, sigerr, A, Aerr, c0, c1, chi2, ndf, False, f"amp snr low ({snr:.2f})")

        return PeakFit(mu, muerr, sig, sigerr, A, Aerr, c0, c1, chi2, ndf, True, "")

    except Exception as e:
        return PeakFit(0,0,0,0,0,0,0,0, np.inf, 0, False, f"fit failed: {e}")

In [74]:
# -----------------------------
# BASELINE DISCOVERY: peak list in ADC channel space
# -----------------------------

def discover_peak_list(spec_x: np.ndarray, spec_y: np.ndarray, spec_yerr: np.ndarray, cfg: dict) -> list[PeakFit]:
    peak_idxs = find_peaks_global(spec_x, spec_y, cfg)
    fits: list[PeakFit] = []
    for idx in peak_idxs:
        pf = fit_peak_gaussian(spec_x, spec_y, spec_yerr, int(idx), cfg)
        if pf.success:
            fits.append(pf)
    # sort by mu ascending
    fits.sort(key=lambda p: p.mu)
    return fits


In [75]:
# -----------------------------
# IDENTIFY CO-60 DOUBLET AUTOMATICALLY (preamp 100)
# -----------------------------

def identify_co60_doublet(peaks: list[PeakFit]) -> tuple[PeakFit, PeakFit] | None:
    """
    Co-60 has two strong lines. In channel space, we don't know spacing a priori,
    but the pair should be:
      - both high amp
      - relatively close compared to full range
      - both good fits

    We'll pick the pair maximizing a simple score:
      score = (A1*A2) / (distance)
    with mild penalty for extreme distance.
    """
    if len(peaks) < 2:
        return None

    best = None
    best_score = -np.inf

    # Only consider top-ish by amplitude to reduce combinatorics
    peaks_sorted = sorted(peaks, key=lambda p: p.amp, reverse=True)[: min(20, len(peaks))]

    for i in range(len(peaks_sorted)):
        for j in range(i+1, len(peaks_sorted)):
            p1, p2 = peaks_sorted[i], peaks_sorted[j]
            d = abs(p2.mu - p1.mu)
            if d <= 0:
                continue
            # avoid pairs that are absurdly close (likely same peak) or absurdly far
            # tune if needed
            if d < 10:
                continue
            score = (abs(p1.amp) * abs(p2.amp)) / d
            if score > best_score:
                best_score = score
                best = (p1, p2)

    if best is None:
        return None
    # order by mu (lower->higher corresponds to 1173->1332 under normal calibration)
    p_lo, p_hi = sorted(best, key=lambda p: p.mu)
    return p_lo, p_hi

In [76]:
# -----------------------------
# TRACKING: find a peak near an expected mu
# -----------------------------

def track_peak_near_mu(x: np.ndarray, y: np.ndarray, yerr: np.ndarray, mu_target: float, cfg: dict) -> PeakFit:
    """
    Search near mu_target, pick best local peak by prominence/SNR, then fit gaussian.
    """
    half = int(cfg["track_search_half_window"])
    # indices window
    idx_center = int(np.argmin(np.abs(x - mu_target)))
    L = max(0, idx_center - half)
    R = min(len(x)-1, idx_center + half)

    xw = x[L:R+1]
    yw = y[L:R+1]

    if len(xw) < cfg["fit_min_points"]:
        return PeakFit(0,0,0,0,0,0,0,0,np.inf,0,False,"track window too small")

    ys = _smooth(yw, CFG["smooth_window"], CFG["smooth_poly"])
    prom = 0.05 * float(np.nanmax(ys) if np.nanmax(ys) > 0 else 1.0)
    cand, props = find_peaks(ys, prominence=prom, distance=cfg["min_distance"])
    if len(cand) == 0:
        # fallback: use max point as candidate
        local_idx = int(np.argmax(ys))
    else:
        # score candidates: prominence / sqrt(local baseline+1) with distance penalty
        prominences = props["prominences"]
        # local baseline estimate
        edge = max(3, len(ys)//10)
        baseline = float(np.median(np.r_[ys[:edge], ys[-edge:]]))
        baseline = max(baseline, 0.0)

        best_score = -np.inf
        best_local = int(cand[0])
        for k, c in enumerate(cand):
            mu_c = xw[c]
            dist = abs(mu_c - mu_target)
            score = float(prominences[k]) / math.sqrt(baseline + 1.0) * math.exp(-(dist**2)/(2*(0.4*half)**2))
            if score > best_score:
                best_score = score
                best_local = int(c)
        local_idx = best_local

    # Convert local index to global index
    idx0 = L + local_idx

    # Fit using a tighter window around this candidate
    fit_cfg = dict(cfg)
    fit_cfg["fit_half_window"] = int(cfg["track_fit_half_window"])
    return fit_peak_gaussian(x, y, yerr, idx0, fit_cfg)



In [77]:
# -----------------------------
# CALIBRATION FITS: E = m*ch + b (or quadratic)
# -----------------------------

def cal_linear(ch, m, b):
    return m*ch + b

def cal_quadratic(ch, a, m, b):
    return a*ch**2 + m*ch + b

def fit_calibration_from_pairs(ch: np.ndarray, E: np.ndarray, ch_err: np.ndarray | None, model: str = "linear") -> CalibrationFit:
    ch = np.asarray(ch, dtype=float)
    E = np.asarray(E, dtype=float)
    if ch_err is None:
        w = np.ones_like(ch)
        Eerr = None
    else:
        # convert channel errors to energy errors iteratively; first pass assume scale ~1
        # We'll iterate once using initial slope.
        # Step 1: unweighted fit to estimate slope
        if model == "linear":
            popt0, _ = curve_fit(cal_linear, ch, E)
            m0, b0 = popt0
            Eerr = np.clip(abs(m0)*ch_err, 1e-12, np.inf)
            popt, pcov = curve_fit(cal_linear, ch, E, sigma=Eerr, absolute_sigma=True)
            yfit = cal_linear(ch, *popt)
            resid = (E - yfit) / Eerr
            chi2 = float(np.sum(resid**2))
            ndf = int(len(ch) - len(popt))
            return CalibrationFit(model="linear", params=np.array(popt), cov=pcov, chi2=chi2, ndf=ndf, used_points=len(ch))

        elif model == "quadratic":
            popt0, _ = curve_fit(cal_quadratic, ch, E)
            a0, m0, b0 = popt0
            # local derivative dE/dch ~ 2a ch + m
            dEdch = np.abs(2*a0*ch + m0)
            Eerr = np.clip(dEdch * ch_err, 1e-12, np.inf)
            popt, pcov = curve_fit(cal_quadratic, ch, E, sigma=Eerr, absolute_sigma=True)
            yfit = cal_quadratic(ch, *popt)
            resid = (E - yfit) / Eerr
            chi2 = float(np.sum(resid**2))
            ndf = int(len(ch) - len(popt))
            return CalibrationFit(model="quadratic", params=np.array(popt), cov=pcov, chi2=chi2, ndf=ndf, used_points=len(ch))

        else:
            raise ValueError("model must be 'linear' or 'quadratic'")

    # If no errors passed, do simple fit
    if model == "linear":
        popt, pcov = curve_fit(cal_linear, ch, E)
        yfit = cal_linear(ch, *popt)
        resid = E - yfit
        # assume unit variance for chi2 proxy
        chi2 = float(np.sum(resid**2))
        ndf = int(len(ch) - len(popt))
        return CalibrationFit(model="linear", params=np.array(popt), cov=pcov, chi2=chi2, ndf=ndf, used_points=len(ch))

    if model == "quadratic":
        popt, pcov = curve_fit(cal_quadratic, ch, E)
        yfit = cal_quadratic(ch, *popt)
        resid = E - yfit
        chi2 = float(np.sum(resid**2))
        ndf = int(len(ch) - len(popt))
        return CalibrationFit(model="quadratic", params=np.array(popt), cov=pcov, chi2=chi2, ndf=ndf, used_points=len(ch))

    raise ValueError("model must be 'linear' or 'quadratic'")

In [78]:
# -----------------------------
# PLOTTING HELPERS
# -----------------------------

def plot_spectrum(x, y, yerr=None, title="", peaks: list[PeakFit] | None = None):
    plt.figure()
    plt.plot(x, y, linewidth=1)
    if peaks:
        for p in peaks:
            plt.axvline(p.mu, linestyle="--", linewidth=1)
    plt.xlabel("ADC Channel")
    plt.ylabel("Counts" if yerr is None else "Net Counts")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_peak_fit(x, y, yerr, pf: PeakFit, title=""):
    plt.figure()
    plt.errorbar(x, y, yerr=yerr, fmt=".", markersize=3)
    if pf.success:
        xx = np.linspace(x.min(), x.max(), 500)
        yy = gaussian_linear(xx, pf.amp, pf.mu, pf.sigma, pf.c0, pf.c1)
        plt.plot(xx, yy, linewidth=2)
        plt.title(f"{title}\nmu={pf.mu:.2f}±{pf.mu_err:.2f}, chi2/ndf={pf.chi2_ndf:.2f}")
    else:
        plt.title(f"{title}\nFIT FAILED: {pf.reason}")
    plt.xlabel("ADC Channel")
    plt.ylabel("Counts")
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_calibration(ch, E, cal: CalibrationFit, title="Calibration"):
    plt.figure()
    plt.scatter(ch, E)
    xx = np.linspace(min(ch), max(ch), 500)
    if cal.model == "linear":
        yy = cal_linear(xx, *cal.params)
    else:
        yy = cal_quadratic(xx, *cal.params)
    plt.plot(xx, yy, linewidth=2)
    plt.xlabel("ADC Channel (centroid)")
    plt.ylabel("Energy (keV)")
    plt.title(f"{title} ({cal.model}), chi2/ndf={cal.chi2_ndf:.2f}")
    plt.grid(True, alpha=0.3)
    plt.show()

    # residuals
    plt.figure()
    if cal.model == "linear":
        pred = cal_linear(np.array(ch), *cal.params)
    else:
        pred = cal_quadratic(np.array(ch), *cal.params)
    res = np.array(E) - pred
    plt.axhline(0, linewidth=1)
    plt.scatter(ch, res)
    plt.xlabel("ADC Channel")
    plt.ylabel("Residual (keV)")
    plt.title(f"{title} residuals")
    plt.grid(True, alpha=0.3)
    plt.show()

In [79]:
# -----------------------------
# FILE DISCOVERY HELPERS
# -----------------------------

def list_csvs(root: Path) -> list[Path]:
    return sorted(root.rglob("*.csv"))

def choose_first_matching(paths: list[Path], must_contain: list[str]) -> Path | None:
    must_contain_low = [s.lower() for s in must_contain]
    for p in paths:
        s = p.as_posix().lower()
        if all(tok in s for tok in must_contain_low):
            return p
    return None


In [80]:
# -----------------------------
# BASELINE CALIBRATION BUILDERS
# -----------------------------

def build_baseline_preamp100(co60_path: Path, na_path: Path | None = None, bg_path: Path | None = None):
    # read
    co60 = read_spectrum_csv(co60_path)
    if bg_path is not None:
        bg = read_spectrum_csv(bg_path)
        net = subtract_background_counts(co60, bg)
        x, y, yerr = net.channel, net.net_counts, net.net_sigma
        title = f"Co60 net (preamp100): {co60_path.name}"
    else:
        x, y, yerr = spectrum_with_errors(co60)
        title = f"Co60 raw (preamp100): {co60_path.name}"

    # discover peaks
    peaks = discover_peak_list(x, y, yerr, CFG)
    plot_spectrum(x, y, title=title, peaks=peaks)

    # identify doublet
    pair = identify_co60_doublet(peaks)
    if pair is None:
        raise RuntimeError("Could not identify Co-60 doublet automatically. Adjust prominence/distance or inspect peaks.")
    p1173, p1332 = pair

    # calibration from two lines (linear)
    ch = np.array([p1173.mu, p1332.mu])
    cherr = np.array([p1173.mu_err, p1332.mu_err])
    E = Co60_spectra
    cal = fit_calibration_from_pairs(ch, E, cherr, model="linear")
    plot_calibration(ch, E, cal, title="Baseline preamp100 (Co60-only)")

    # optional: validate/augment with Na
    if na_path is not None:
        na = read_spectrum_csv(na_path)
        if bg_path is not None:
            bg = read_spectrum_csv(bg_path)
            net_na = subtract_background_counts(na, bg)
            xn, yn, en = net_na.channel, net_na.net_counts, net_na.net_sigma
        else:
            xn, yn, en = spectrum_with_errors(na)

        # predict channels for Na lines using current cal, invert approx by scanning:
        # We'll track peaks near expected mu by mapping E->ch using inverse linear:
        m, b = cal.params
        mu_targets = (Na_spectra - b) / m  # channel targets
        na_fits = []
        for mut in mu_targets:
            pf = track_peak_near_mu(xn, yn, en, float(mut), CFG)
            na_fits.append(pf)

        # keep successful fits
        good = [pf for pf in na_fits if pf.success]
        if len(good) >= 1:
            plot_spectrum(xn, yn, title=f"Na spectrum with tracked peaks: {na_path.name}", peaks=good)

        # If both Na peaks found, refit calibration with 4 points (Co60 + Na)
        if sum(pf.success for pf in na_fits) == 2:
            ch4 = np.array([p1173.mu, p1332.mu, na_fits[0].mu, na_fits[1].mu])
            ch4e = np.array([p1173.mu_err, p1332.mu_err, na_fits[0].mu_err, na_fits[1].mu_err])
            E4 = np.array([Co60_spectra[0], Co60_spectra[1], Na_spectra[0], Na_spectra[1]])
            cal4 = fit_calibration_from_pairs(ch4, E4, ch4e, model=CFG["cal_model"])
            plot_calibration(ch4, E4, cal4, title=f"Baseline preamp100 (Co60+Na, {CFG['cal_model']})")
            return cal4, {"co60_doublet": (p1173, p1332), "na_peaks": na_fits}

    return cal, {"co60_doublet": (p1173, p1332)}

def build_baseline_preamp0(cs_path: Path, ba_path: Path | None = None, bg_path: Path | None = None):
    """
    For preamp0:
      - auto-detect strong Cs peak (likely 661)
      - if Ba provided, use pattern: predict Ba lines after initial linear guess and harvest them
    """
    cs = read_spectrum_csv(cs_path)
    if bg_path is not None:
        bg = read_spectrum_csv(bg_path)
        net = subtract_background_counts(cs, bg)
        x, y, yerr = net.channel, net.net_counts, net.net_sigma
        title = f"Cs net (preamp0): {cs_path.name}"
    else:
        x, y, yerr = spectrum_with_errors(cs)
        title = f"Cs raw (preamp0): {cs_path.name}"

    peaks = discover_peak_list(x, y, yerr, CFG)
    plot_spectrum(x, y, title=title, peaks=peaks)

    # pick strongest peak as Cs 661 candidate (often true)
    if len(peaks) == 0:
        raise RuntimeError("No peaks found in Cs spectrum. Adjust peak-finding thresholds.")
    cs_candidate = max(peaks, key=lambda p: p.amp)

    # Without a second anchor, we can't fully calibrate from Cs alone.
    # If Ba is available, we use Ba to establish slope; otherwise we return just Cs centroid.
    info = {"cs_661_candidate": cs_candidate}

    if ba_path is None:
        return None, info

    ba = read_spectrum_csv(ba_path)
    if bg_path is not None:
        bg = read_spectrum_csv(bg_path)
        netb = subtract_background_counts(ba, bg)
        xb, yb, eb = netb.channel, netb.net_counts, netb.net_sigma
    else:
        xb, yb, eb = spectrum_with_errors(ba)

    # Discover Ba peaks globally
    ba_peaks = discover_peak_list(xb, yb, eb, CFG)
    plot_spectrum(xb, yb, title=f"Ba spectrum peaks: {ba_path.name}", peaks=ba_peaks)

    # AUTOMATED MATCH (simple, practical):
    # Use one strong Ba line (356 keV often strong) as second anchor by searching for a strong Ba peak
    # in the upper-middle channel region relative to Cs peak.
    #
    # Heuristic: choose a Ba peak that is at lower channel than Cs peak (since 356 < 661) but not tiny.
    # Then build linear calibration from (Ba356, Cs661), and harvest more Ba lines by tracking.
    #
    # You can tune or replace this matcher with a RANSAC-style matcher later.
    cs_mu = cs_candidate.mu
    candidates = [p for p in ba_peaks if p.mu < cs_mu and p.amp > 0]
    if len(candidates) == 0:
        raise RuntimeError("Could not find a usable Ba peak below Cs peak. Try different Ba run or adjust thresholds.")
    # pick best candidate near ~ (cs_mu * 356/661) if roughly linear through origin
    target = cs_mu * (356.005 / 661.657)
    ba356 = min(candidates, key=lambda p: abs(p.mu - target))
    info["ba356_candidate"] = ba356

    ch2 = np.array([ba356.mu, cs_candidate.mu])
    ch2e = np.array([ba356.mu_err, cs_candidate.mu_err])
    E2 = np.array([356.005, 661.657])
    cal2 = fit_calibration_from_pairs(ch2, E2, ch2e, model="linear")
    plot_calibration(ch2, E2, cal2, title="Baseline preamp0 (Ba356 + Cs661)")

    # Harvest more Ba lines by tracking around predicted channels
    m, b = cal2.params
    mu_targets = (Ba_spectra - b) / m

    tracked = []
    for mut in mu_targets:
        pf = track_peak_near_mu(xb, yb, eb, float(mut), CFG)
        tracked.append(pf)

    good = [pf for pf in tracked if pf.success]
    info["ba_tracked"] = tracked

    if len(good) >= 2:
        # Fit calibration using all good Ba + Cs
        ch = np.array([pf.mu for pf in good] + [cs_candidate.mu])
        cherr = np.array([pf.mu_err for pf in good] + [cs_candidate.mu_err])

        # energies: keep only those Ba energies whose tracked peaks succeeded
        E_good = np.array([Ba_spectra[i] for i, pf in enumerate(tracked) if pf.success] + [661.657])

        cal = fit_calibration_from_pairs(ch, E_good, cherr, model=CFG["cal_model"])
        plot_calibration(ch, E_good, cal, title=f"Baseline preamp0 (Ba+Cs, {CFG['cal_model']})")
        return cal, info

    return cal2, info


In [81]:
# -----------------------------
# DAILY DRIFT ESTIMATION
# -----------------------------

def drift_preamp0_shift(daily_cs_path: Path, baseline_cs_mu: float, bg_path: Path | None = None) -> dict:
    cs = read_spectrum_csv(daily_cs_path)
    if bg_path is not None:
        bg = read_spectrum_csv(bg_path)
        net = subtract_background_counts(cs, bg)
        x, y, yerr = net.channel, net.net_counts, net.net_sigma
    else:
        x, y, yerr = spectrum_with_errors(cs)

    pf = track_peak_near_mu(x, y, yerr, baseline_cs_mu, CFG)
    if not pf.success:
        return {"mode": "none", "reason": f"Cs tracking failed: {pf.reason}"}

    shift = baseline_cs_mu - pf.mu  # add this to today's channels to align to baseline
    return {"mode": "shift", "shift": shift, "daily_cs_mu": pf.mu, "daily_cs_mu_err": pf.mu_err, "fit": pf}

def drift_preamp100_affine(daily_co60_path: Path, baseline_pair_mus: tuple[float, float], bg_path: Path | None = None) -> dict:
    co60 = read_spectrum_csv(daily_co60_path)
    if bg_path is not None:
        bg = read_spectrum_csv(bg_path)
        net = subtract_background_counts(co60, bg)
        x, y, yerr = net.channel, net.net_counts, net.net_sigma
    else:
        x, y, yerr = spectrum_with_errors(co60)

    mu0, mu1 = baseline_pair_mus
    p0 = track_peak_near_mu(x, y, yerr, mu0, CFG)
    p1 = track_peak_near_mu(x, y, yerr, mu1, CFG)

    if p0.success and p1.success:
        # Solve baseline_mu = a * daily_mu + b
        # from two points
        d0, d1 = p0.mu, p1.mu
        A = np.array([[d0, 1.0], [d1, 1.0]], dtype=float)
        bvec = np.array([mu0, mu1], dtype=float)
        a, b_ = np.linalg.solve(A, bvec)
        return {"mode": "affine", "a": a, "b": b_, "daily_mus": (p0.mu, p1.mu), "fits": (p0, p1)}

    # fallback: shift-only using whichever peak succeeded
    if p0.success:
        shift = mu0 - p0.mu
        return {"mode": "shift", "shift": shift, "used": "low", "fit": p0, "other_failed": p1.reason}
    if p1.success:
        shift = mu1 - p1.mu
        return {"mode": "shift", "shift": shift, "used": "high", "fit": p1, "other_failed": p0.reason}

    return {"mode": "none", "reason": f"Co60 tracking failed: low({p0.reason}), high({p1.reason})"}

def apply_drift_correction_channels(ch: np.ndarray, drift: dict) -> np.ndarray:
    ch = np.asarray(ch, dtype=float)
    if drift["mode"] == "shift":
        return ch + float(drift["shift"])
    if drift["mode"] == "affine":
        return float(drift["a"]) * ch + float(drift["b"])
    return ch.copy()

In [82]:
# -----------------------------
# RUN
# -----------------------------

all_csvs = list_csvs(ROOT_CAL)
print(f"Found {len(all_csvs)} CSVs under {ROOT_CAL}")

# ---- baseline selection (edit tokens to match your names) ----
baseline_cs = choose_first_matching(all_csvs, ["feb_24", "preamp_0", "cs"])  # e.g. processed_data/calibration/feb_24/preamp_0/Cs_...
baseline_ba = choose_first_matching(all_csvs, ["feb_24", "preamp_0", "ba"])
baseline_co60 = choose_first_matching(all_csvs, ["feb_24", "preamp_100", "co60"])
baseline_na = choose_first_matching(all_csvs, ["feb_24", "preamp_100", "na"])

print("baseline_cs:", baseline_cs)
print("baseline_ba:", baseline_ba)
print("baseline_co60:", baseline_co60)
print("baseline_na:", baseline_na)

# Background path (optional)
bg_path = BG_DEFAULT

# ---- build baseline calibrations ----
cal0, info0 = (None, {})
if baseline_cs is not None:
    cal0, info0 = build_baseline_preamp0(baseline_cs, baseline_ba, bg_path=bg_path)

cal100, info100 = (None, {})
if baseline_co60 is not None:
    cal100, info100 = build_baseline_preamp100(baseline_co60, baseline_na, bg_path=bg_path)

print("Baseline preamp0 cal:", cal0.params if cal0 else None)
print("Baseline preamp100 cal:", cal100.params if cal100 else None)

# ---- daily examples (edit for your daily filenames) ----
daily_cs = choose_first_matching(all_csvs, ["daily", "cs", "preamp0"])
daily_co60 = choose_first_matching(all_csvs, ["daily", "co60", "preamp100"])

print("daily_cs:", daily_cs)
print("daily_co60:", daily_co60)

# ---- compute drift and show ----
if daily_cs is not None and "cs_661_candidate" in info0:
    baseline_cs_mu = info0["cs_661_candidate"].mu
    drift0 = drift_preamp0_shift(daily_cs, baseline_cs_mu, bg_path=bg_path)
    print("drift preamp0:", drift0)

if daily_co60 is not None and "co60_doublet" in info100:
    p_lo, p_hi = info100["co60_doublet"]
    drift100 = drift_preamp100_affine(daily_co60, (p_lo.mu, p_hi.mu), bg_path=bg_path)
    print("drift preamp100:", drift100)

# After this:
# 1) apply drift correction to any spectrum’s channel axis
# 2) apply baseline calibration E(ch) for conversion / ROI integrations / etc.
#
# Example:
#   ch_corr = apply_drift_correction_channels(spec.channel, drift0)
#   E = cal_linear(ch_corr, *cal0.params)  # if cal0.model == "linear"
#
# (Or use quadratic if CFG["cal_model"] == "quadratic".)

Found 10 CSVs under processed_data/calibration
baseline_cs: processed_data/calibration/feb_24/preamp_0/Ba_22.4C_preamp0_feb24.csv
baseline_ba: processed_data/calibration/feb_24/preamp_0/Ba_22.4C_preamp0_feb24.csv
baseline_co60: processed_data/calibration/feb_24/preamp_100/Co60_22.5C_preamp100_feb24.csv
baseline_na: processed_data/calibration/feb_24/preamp_100/Na_22.5C_preamp100_feb24.csv


ValueError: Bad dt (t_sig=nan, t_bg=nan). Ensure elapsed time or start/end times parsed correctly.